# Pipeline de Pré-Processamento — DeepLabCut vs. TopScan
**Laboratório de Memória - Instituto do Cérebro (UFRN) | Validação Técnica de Rastreamento Animal**

---

## Visão Geral

Este notebook realiza o pré-processamento dos dados brutos de rastreamento antes da análise estatística.
Ele deve ser executado **antes** do notebook `Pipeline de Análise — DeepLabCut vs. TopScan.ipynb`.

**Fluxo da pipeline:**
```
Vídeos .MPG  →  Conversão .MP4  →  Recorte por animal
     ↓
Arquivos .H5 (DLC)  →  Filtro de Likelihood (> 0.9)  →  TXT por bodypart
     ↓
TXT TopScan (bruto)  →  Remoção de -1  →  IQR + DBSCAN  →  Savitzky-Golay  →  Calibração (Regressão)  →  TXT final
```

---

## Como Executar

Execute as células **em ordem, de cima para baixo**. Cada seção é independente após o mount do Drive.

| Seção | O que faz | Precisa configurar? |
|---|---|---|
| Instalação de Dependências | Instala ffmpeg, scipy, scikit-learn | Não |
| Conversão de Vídeos | .MPG → .MP4 com ffmpeg | Sim — `input_folder`, `output_folder`, `initial` |
| Recorte de Vídeos | Divide vídeo multi-animal por quadrante | Sim — `pasta_videos` |
| Exportação H5 → TXT | Extrai coordenadas DLC com filtro de likelihood | Sim — `experimento`, `bodypart_selecionado` |
| Calibração TopScan | Calibra, filtra outliers e mescla eventos | Sim — `base_path`, `saida_path_dir` |

---

## Parâmetros que Você Deve Ajustar

```python
# --- Seção DLC ---
experimento          = "experimento"   # Prefixo dos arquivos .h5
bodypart_selecionado = 'tail_tip'      # Bodypart de interesse (ver bodypart_mapping)
LIMIAR_LIKELIHOOD    = 0.9             # Frames abaixo disso são descartados como NaN

# --- Seção TopScan ---
base_path      = "...TXT original e XLSX original/"  # Pasta com os .TXT e .XLSX brutos
saida_path_dir = "...TXT novo/"                      # Pasta de saída dos arquivos calibrados
```

> **Dica:** Nunca altere `LIMIAR_LIKELIHOOD` para valores abaixo de `0.85`.
> Predições com likelihood baixo indicam oclusão ou falha do modelo ResNet-50.

---

## Estrutura Esperada no Google Drive

```
MyDrive/
└── validacao_topscan/
    ├── DEEPLABCUT/
    │   ├── h5_validacao/          ← arquivos .h5 exportados pelo DLC
    │   └── trajetoria_txt_dlc/    ← TXTs gerados por este notebook
    └── TOPSCAN/
        ├── TXT original e XLSX original/  ← arquivos brutos do TopScan
        └── TXT novo/                      ← TXTs calibrados gerados por este notebook
```

---

## Sobre os Filtros Aplicados

- **IQR (deslocamento):** Remove saltos frame-a-frame fisicamente impossíveis. Limiar padrão: `Q3 + 2.5 * IQR`.
- **DBSCAN:** Remove clusters de predição isolados (glitches). Parâmetro `eps=15 px` — ajustar se a resolução do vídeo for muito diferente de ~40 px/cm.
- **Savitzky-Golay:** Suaviza ruído de alta frequência sem distorcer picos de velocidade. Janela padrão: `11 frames` (~0.37 s a 30 fps).
- **Regressão Huber:** Calibra TopScan → espaço DLC com intercepto (corrige translação). R² < 0.90 indica distorção de lente — reportar ao responsável pela configuração da câmera.

---

# CONEXÃO COM O SEU DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## INSTALAÇÃO DE DEPENDÊNCIAS

In [ ]:
!apt-get install -q ffmpeg
!pip install -q ffmpeg-python moviepy
!pip install -q --upgrade pandas scikit-learn scipy

## IMPORTS

In [ ]:
import os
import re
import subprocess

import cv2
import ffmpeg
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.linear_model import HuberRegressor
from sklearn.metrics import r2_score
from sklearn.cluster import DBSCAN
from tqdm import tqdm

## CONVERSÃO DE VÍDEOS .MPG PARA .MP4 (TOPSCAN)

In [ ]:
input_folder  = '/content/drive/My Drive/TESTES_MPG'
output_folder = '/content/drive/My Drive/TESTES_MP4'
initial = ('Moti_EI_R35', 'Moti_EI_R12_D7')

os.makedirs(output_folder, exist_ok=True)

def convert_videos(input_folder, output_folder):
    files = [f for f in os.listdir(input_folder)
             if f.lower().endswith('.mpg') and f.startswith(initial)]

    with tqdm(total=len(files), desc="Convertendo vídeos", unit="vídeo") as pbar:
        for filename in files:
            input_path  = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, os.path.splitext(filename)[0] + '.mp4')

            try:
                try:
                    duration = float(ffmpeg.probe(input_path)['format']['duration'])
                except ffmpeg.Error as e:
                    print(f'\nErro ao ler {filename}: {e.stderr.decode("utf-8")}')
                    pbar.update(1)
                    continue

                command = ['ffmpeg', '-i', input_path, '-r', '30', output_path]
                process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)

                last_progress = -1
                while True:
                    line = process.stderr.readline()
                    if not line and process.poll() is not None:
                        break
                    if "time=" in line:
                        match = re.search(r'time=(\d+:\d+:\d+\.\d+)', line)
                        if match:
                            current_time = sum(float(x) * 60 ** i for i, x in enumerate(reversed(match.group(1).split(':'))))
                            progress = (current_time / duration) * 100
                            if int(progress) != last_progress:
                                print(f'\rConvertendo {filename}: {int(progress)}% concluído', end='')
                                last_progress = int(progress)

                process.wait()
                print(f'\nConvertido {filename}')
                pbar.update(1)

            except Exception as e:
                print(f'\nErro ao converter {filename}: {e}\n')
                pbar.update(1)

convert_videos(input_folder, output_folder)
print("Conversão concluída!")


## RECORTE DE VÍDEOS MULTI-ANIMAL

In [ ]:
def cortar_e_salvar_com_barra(video_path, saida_base, largura_barra=35):
    nome_arquivo      = os.path.basename(video_path)
    nome_sem_extensao = os.path.splitext(nome_arquivo)[0]
    ids               = nome_sem_extensao.split("_")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Erro ao abrir: {video_path}")
        return

    fps          = cap.get(cv2.CAP_PROP_FPS)
    frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    codec        = cv2.VideoWriter_fourcc(*'mp4v')

    coords = [
        (35, 0,                   frame_width // 2 + 5,  frame_height // 2),
        (frame_width // 2 + 30,   0,                      frame_width - 20,  frame_height // 2),
        (35, frame_height // 2 + 5, frame_width // 2 + 5, frame_height - 10),
    ]

    saidas = {}
    for idx in range(min(len(ids), len(coords))):
        x1, y1, x2, y2 = coords[idx]
        saida_path      = os.path.join(saida_base, f"{ids[idx].strip()}.mp4")
        saidas[idx]     = cv2.VideoWriter(saida_path, codec, fps, (x2 - x1, y2 - y1))

    cor_barra = (57, 60, 60)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        for idx, writer in saidas.items():
            x1, y1, x2, y2 = coords[idx]
            crop = frame[y1:y2, x1:x2].copy()
            crop[:, -largura_barra:] = cor_barra
            writer.write(crop)

    cap.release()
    for writer in saidas.values():
        writer.release()
    print(f"Finalizado: {nome_arquivo} -> {len(saidas)} vídeos gerados.")

pasta_videos = "/content/drive/MyDrive/TESTES_MP4"
saida_base   = os.path.join(pasta_videos, "videos_processados")
os.makedirs(saida_base, exist_ok=True)

for arquivo in os.listdir(pasta_videos):
    if arquivo.endswith(".mp4") and "_" in arquivo:
        cortar_e_salvar_com_barra(os.path.join(pasta_videos, arquivo), saida_base)

## EXPORTAÇÃO DOS DADOS H5 --> TXT (DEEPLABCUT) COM FILTRO DE LIKELIHOOD E RELATÓRIO DE QUALIDADE

In [ ]:
# =================================================================
# CONFIGURAÇÕES DO USUÁRIO
# =================================================================
experimento          = "*"        # Filtro de início de arquivo
bodypart_selecionado = 'body'     # Parte do corpo desejada
LIMIAR_LIKELIHOOD    = 0.9        # Frames abaixo disso viram NaN
# =================================================================

bodypart_mapping = {
    'nose': 'nose', 'neck': 'neck', 'body': 'body',
    'tail_base': 'tail_base', 'tail_middle': 'tail_middle', 'tail_tip': 'tail_tip',
}

diretorio_h5el = '/content/drive/MyDrive/H5'
diretorio_txt  = '/content/drive/MyDrive/TXT'
os.makedirs(diretorio_txt, exist_ok=True)

def extrair_id_universal(nome_arquivo):
    match = re.search(r'^(.*?)DLC', nome_arquivo, re.IGNORECASE)
    if match:
        return match.group(1).strip('_')
    return "X"

# Lista de ficheiros e setup para o relatório de controlo de qualidade (QC)
arquivos = [f for f in os.listdir(diretorio_h5el) if f.lower().endswith(".h5")]
qc_report = []

for arquivo in arquivos:
    if experimento != "*" and not arquivo.startswith(experimento):
        continue

    caminho_h5 = os.path.join(diretorio_h5el, arquivo)
    rat_id     = extrair_id_universal(arquivo)

    print(f"\n--- Lendo: {arquivo} ---")
    print(f"ID Identificado: {rat_id}")

    try:
        df = pd.read_hdf(caminho_h5)
        scorer = df.columns.get_level_values(0)[0]

        data_bp = df[scorer][bodypart_selecionado]
        x = data_bp['x']
        y = data_bp['y']
        likelihood = data_bp['likelihood']

        mascara_baixa_conf = likelihood < LIMIAR_LIKELIHOOD
        x = x.where(~mascara_baixa_conf, other=np.nan)
        y = y.where(~mascara_baixa_conf, other=np.nan)

        n_frames = len(df)
        n_rej = int(mascara_baixa_conf.sum())

        print(f"Rejeitados: {n_rej}/{n_frames} ({n_rej/n_frames*100:.1f}%)")

        nome_txt = f"ID_{rat_id}_trajetoria_{bodypart_selecionado}.txt"
        caminho_saida = os.path.join(diretorio_txt, nome_txt)

        pd.DataFrame({
            'Frame': range(n_frames),
            'X': x.round(2),
            'Y': y.round(2),
        }).to_csv(caminho_saida, sep=' ', index=False, float_format='%.2f', na_rep='NaN')

        print(f"Sucesso: {nome_txt} salvo.")

        # Guarda os dados para o ficheiro de controlo de qualidade
        qc_report.append({
            'Animal_ID': rat_id,
            'Ficheiro': arquivo,
            'Total_Frames': n_frames,
            'Frames_Rejeitados': n_rej,
            'Perda_Percentual': round((n_rej/n_frames)*100, 2)
        })

    except KeyError:
        print(f"ERRO: Bodypart '{bodypart_selecionado}' não encontrada no arquivo.")
    except Exception as e:
        print(f"ERRO inesperado: {e}")

df_qc = pd.DataFrame(qc_report)
caminho_qc = os.path.join(diretorio_txt, 'QC_Relatorio_DLC.csv')
df_qc.to_csv(caminho_qc, index=False)

print(f"\n--- Relatório de Controlo de Qualidade (QC) salvo em: {caminho_qc} ---")
print("--- Processo Finalizado ---")

## FUNÇÕES DE PRÉ-PROCESSAMENTO: OUTLIERS E SUAVIZAÇÃO

In [ ]:
def filtrar_outliers_trajetoria(df, col_x='X', col_y='Y', eps=15.0, min_samples=3, iqr_multiplicador=2.5):
    df = df.copy()
    dx = np.diff(df[col_x].ffill().values, prepend=df[col_x].iloc[0])
    dy = np.diff(df[col_y].ffill().values, prepend=df[col_y].iloc[0])
    deslocamento = np.sqrt(dx**2 + dy**2)

    span_x = np.nanpercentile(df[col_x], 95) - np.nanpercentile(df[col_x], 5)
    span_y = np.nanpercentile(df[col_y], 95) - np.nanpercentile(df[col_y], 5)
    p95 = np.nanpercentile(deslocamento, 95) if np.isfinite(deslocamento).any() else 0.0
    baixa_mobilidade = (p95 < 3.0) or (span_x < 20.0 and span_y < 20.0)

    Q1, Q3  = np.nanpercentile(deslocamento, 25), np.nanpercentile(deslocamento, 75)
    limiar  = Q3 + iqr_multiplicador * (Q3 - Q1)
    if baixa_mobilidade:
        limiar = max(limiar, 15.0)
        print("  Modo baixa mobilidade: limiar de salto relaxado e DBSCAN desativado.")
    saltos  = deslocamento > limiar
    df.loc[saltos, [col_x, col_y]] = np.nan
    print(f"  IQR: {saltos.sum()} saltos removidos (limiar: {limiar:.1f} px/frame)")

    idx_validos     = df[[col_x, col_y]].dropna().index
    coords_validos  = df.loc[idx_validos, [col_x, col_y]].values

    if (not baixa_mobilidade) and len(coords_validos) > min_samples:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(coords_validos)
        idx_ruido = idx_validos[labels == -1]
        df.loc[idx_ruido, [col_x, col_y]] = np.nan
        print(f"  DBSCAN: {(labels == -1).sum()} pontos de ruído removidos")

    return df

def suavizar_trajetoria(df, col_x='X', col_y='Y', janela=11, ordem=3):
    df = df.copy()
    for col in [col_x, col_y]:
        serie        = df[col].copy()
        # AUMENTADO O LIMITE PARA 15 (aprox. 0.5 seg a 30fps) para lidar com gaps na suavização
        serie_interp = serie.interpolate(method='linear', limit_direction='forward', limit=15)
        n_validos    = serie_interp.notna().sum()

        if n_validos >= janela:
            valores_preenchidos = serie_interp.ffill().bfill()

            valores_sg = savgol_filter(
                valores_preenchidos,
                window_length=janela, polyorder=ordem,
            )
            df[col] = np.where(serie.isna(), np.nan, valores_sg)
        else:
            print(f"  Aviso: {col} com apenas {n_validos} pontos válidos — SG não aplicado.")

    return df

## CRIAÇÃO DE TXT SENSÍVEL (TOPSCAN) COM CALIBRAÇÃO POR REGRESSÃO

In [ ]:
# --- 1. Configuração dos Diretórios ---
base_path         = "/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/TOPSCAN/TXT-XLSX_ORIGINAL/"
saida_path_dir    = "/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/TOPSCAN/TXT_NOVO/"
diretorio_dlc_ref = "/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/DLC/TXT"

# --- 2. Diagnóstico de Alinhamento Temporal ---
def diagnosticar_alinhamento_temporal(df_ts, df_dlc, col_frame='FrameNum'):
    print("\n  --- Diagnóstico Temporal ---")
    for nome, df in [('TopScan', df_ts), ('DLC', df_dlc)]:
        frames = df[col_frame].sort_values()
        n_dupl = int(frames.duplicated().sum())
        gaps   = frames.diff().dropna()
        n_gaps = int((gaps > 1).sum())
        if n_dupl:
            print(f"  {nome}: {n_dupl} frames DUPLICADOS")
        if n_gaps:
            print(f"  {nome}: {n_gaps} gaps na sequência de frames")
        if not n_dupl and not n_gaps:
            print(f"  {nome}: sequência contínua ({int(frames.min())}–{int(frames.max())})")

    frames_ts  = set(df_ts[col_frame])
    frames_dlc = set(df_dlc[col_frame])
    overlap    = len(frames_ts & frames_dlc)
    pct        = overlap / max(len(frames_ts), len(frames_dlc)) * 100
    print(f"  Sobreposição: {overlap} frames ({pct:.1f}%) | TS={len(frames_ts)}, DLC={len(frames_dlc)}")
    return overlap

# --- 3. Calibração por Regressão Linear Robusta (Huber) ---
def calibrar_por_regressao(df_limpo_ts, df_dlc_ref,
                            col_ts_x='CenterX(mm)', col_ts_y='CenterY(mm)'):
    df_merged = pd.merge(df_limpo_ts, df_dlc_ref, on='FrameNum', how='inner').dropna()

    if len(df_merged) < 30:
        print("  Aviso: pontos insuficientes para regressão. Pulando calibração.")
        return df_limpo_ts.copy(), None

    diagnosticar_alinhamento_temporal(df_limpo_ts, df_dlc_ref)

    df_calibrado = df_limpo_ts.copy()
    modelos      = {}

    for eixo_ts, eixo_dlc in [(col_ts_x, 'X_dlc'), (col_ts_y, 'Y_dlc')]:
        X      = df_merged[[eixo_ts]].values
        y_alvo = df_merged[eixo_dlc].values

        modelo = HuberRegressor(epsilon=1.5).fit(X, y_alvo)
        r2     = r2_score(y_alvo, modelo.predict(X))

        print(f"  Regressão {eixo_ts}: coef={modelo.coef_[0]:.5f}, intercept={modelo.intercept_:.3f}, R²={r2:.4f}")
        if r2 < 0.90:
            print(f"  Aviso: R²={r2:.3f} baixo em {eixo_ts}. Verificar alinhamento câmera/arena.")

        df_calibrado[eixo_ts] = modelo.coef_[0] * df_calibrado[eixo_ts] + modelo.intercept_
        modelos[eixo_ts] = {'coef': modelo.coef_[0], 'intercept': modelo.intercept_, 'r2': r2}

    return df_calibrado, modelos

# --- 4. Função Principal de Processamento ---
def processar_e_calibrar_topscan(txt_path, xlsx_path, saida_path):
    print(f"\nIniciando processo para {os.path.basename(txt_path)}...")

    try:
        colunas_txt      = ['FrameNum', 'CenterX(mm)', 'CenterY(mm)', 'Areas']
        df_trajetoria_ts = pd.read_csv(txt_path, sep=r'\s+', skiprows=1, names=colunas_txt)
        df_trajetoria_ts['Areas'] = df_trajetoria_ts['Areas'].astype(str).str.replace(',', '', regex=False)
    except Exception as e:
        print(f"  ERRO ao ler TXT do TopScan: {e}")
        return

    df_trajetoria_ts['CenterX(mm)'] = pd.to_numeric(df_trajetoria_ts['CenterX(mm)'], errors='coerce')
    df_trajetoria_ts['CenterY(mm)'] = pd.to_numeric(df_trajetoria_ts['CenterY(mm)'], errors='coerce')

    df_limpo_ts = df_trajetoria_ts[
        (df_trajetoria_ts['CenterX(mm)'] > 0) & (df_trajetoria_ts['CenterY(mm)'] > 0)
    ].copy()
    print(f"  TopScan: {len(df_limpo_ts)} linhas válidas identificadas.")

    df_limpo_ts = filtrar_outliers_trajetoria(df_limpo_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')
    df_limpo_ts = suavizar_trajetoria(df_limpo_ts, col_x='CenterX(mm)', col_y='CenterY(mm)')

    nome_arq = os.path.basename(txt_path)
    nome_base = os.path.splitext(nome_arq)[0]

    match_composto = re.search(r'([A-Za-z\-]+)_(\d+)', nome_base)
    rat_id = None
    indice_real = 0
    if match_composto:
        sequencia_pares = match_composto.group(1).split('-')
        indice_real = int(match_composto.group(2)) - 1
        if 0 <= indice_real < len(sequencia_pares):
            rat_id = sequencia_pares[indice_real]

    CAMPO_LARGURA_PX = 720
    CAMPO_ALTURA_PX  = 480
    _offsets = {
        0: (0,                0),
        1: (CAMPO_LARGURA_PX, 0),
        2: (0,                CAMPO_ALTURA_PX),
    }
    _x_off, _y_off = _offsets.get(indice_real, (0, 0))
    df_limpo_ts = df_limpo_ts.copy()
    df_limpo_ts['CenterX(mm)'] -= _x_off
    df_limpo_ts['CenterY(mm)'] -= _y_off

    if not rat_id:
        match_simples = re.search(r'^([A-Za-z]+)', nome_base)
        if match_simples:
            rat_id = match_simples.group(1)

    if not rat_id:
        df_calibrado_ts = df_limpo_ts.copy()
    else:
        candidatos_dlc = [
            f"ID_{rat_id}_trajetoria_body.txt",
            f"ID_{rat_id}_trajetoria_centerbody.txt",
            f"ID_{rat_id}_trajetoria_nose.txt",
            f"{rat_id}_trajetoria_body.txt"
        ]
        path_dlc_ref = None
        for candidato in candidatos_dlc:
            caminho_teste = os.path.join(diretorio_dlc_ref, candidato)
            if os.path.exists(caminho_teste):
                path_dlc_ref = caminho_teste
                break

        if not path_dlc_ref:
            df_calibrado_ts = df_limpo_ts.copy()
        else:
            try:
                df_dlc_ref = pd.read_csv(path_dlc_ref, sep=r'\s+').rename(
                    columns={'Frame': 'FrameNum', 'X': 'X_dlc', 'Y': 'Y_dlc'}
                )
                df_dlc_ref = df_dlc_ref.dropna(subset=['X_dlc', 'Y_dlc'])
                df_calibrado_ts, _ = calibrar_por_regressao(df_limpo_ts, df_dlc_ref)
            except Exception as e:
                print(f"  ERRO durante calibração: {e}")
                df_calibrado_ts = df_limpo_ts.copy()

    try:
        df_eventos = pd.read_excel(xlsx_path, header=1)
        df_eventos.columns = df_eventos.columns.str.strip()

        df_mesclado_final = df_calibrado_ts.copy()
        df_mesclado_final['Areas'] = 'Floor'

        for _, row in df_eventos.iterrows():
            f_inicio = int(row['From Frame'])
            f_fim    = int(row['To Frame'])
            evento   = str(row['Event'])
            evento_limpo = evento.split("sniffing On")[-1].strip() if "sniffing On" in evento else evento.strip()

            df_mesclado_final.loc[
                (df_mesclado_final['FrameNum'] >= f_inicio) &
                (df_mesclado_final['FrameNum'] <= f_fim), 'Areas'
            ] = evento_limpo
    except Exception as e:
        print(f"  ERRO ao processar Excel: {e}")
        return

    os.makedirs(os.path.dirname(saida_path), exist_ok=True)
    with open(saida_path, 'w') as f_out:
        f_out.write("FrameNum CenterX(mm) CenterY(mm) Areas\n")
        df_mesclado_final.to_csv(f_out, sep=' ', index=False, header=False, float_format='%.2f', na_rep='NaN')
    print(f"  Sucesso: {os.path.basename(saida_path)} salvo.")

def executar_processo_automatico():
    if not os.path.isdir(base_path):
        return
    arquivos_txt = [f for f in os.listdir(base_path) if f.upper().endswith('.TXT')]
    for f_txt in sorted(arquivos_txt):
        nome_base  = os.path.splitext(f_txt)[0]
        txt_path   = os.path.join(base_path, f_txt)
        xlsx_path  = os.path.join(base_path, f"{nome_base}.xlsx")
        saida_path = os.path.join(saida_path_dir, f"{nome_base}_NOVO.TXT")

        if os.path.exists(xlsx_path):
            processar_e_calibrar_topscan(txt_path, xlsx_path, saida_path)

executar_processo_automatico()

In [ ]:
DIRETORIO_TXT_NOVO = '/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/TOPSCAN/TXT_NOVO/'
DIRETORIO_RESULTADOS = '/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/RESULTADOS/'
FPS = 30

def gerar_relatorio_tempos():
    if not os.path.isdir(DIRETORIO_TXT_NOVO):
        return None

    arquivos = sorted([f for f in os.listdir(DIRETORIO_TXT_NOVO) if f.upper().endswith('.TXT')])
    resultados = []

    for arquivo in arquivos:
        caminho = os.path.join(DIRETORIO_TXT_NOVO, arquivo)
        try:
            df = pd.read_csv(caminho, sep=r'\s+')
            contagem_frames = df['Areas'].value_counts()
            nome_base = arquivo.replace('_NOVO.TXT', '')
            dados_video = {'Vídeo / Par': nome_base}

            for area, frames in contagem_frames.items():
                if str(area).lower() != 'floor' and str(area) != 'nan':
                    tempo_segundos = frames / FPS
                    dados_video[area] = round(tempo_segundos, 2)

            resultados.append(dados_video)
        except Exception as e:
            pass

    if resultados:
        df_resultados = pd.DataFrame(resultados).fillna(0)
        cols = ['Vídeo / Par'] + [col for col in df_resultados.columns if col != 'Vídeo / Par']
        df_resultados = df_resultados[cols]
        display(df_resultados)
        os.makedirs(DIRETORIO_RESULTADOS, exist_ok=True)
        caminho_excel = os.path.join(DIRETORIO_RESULTADOS, 'Resumo_Tempos_TopScan_Extraidos.xlsx')
        df_resultados.to_excel(caminho_excel, index=False)
        return df_resultados
    else:
        return None

df_resultados = gerar_relatorio_tempos()

In [ ]:
tempos_manuais = {
    "AA": {"Esquerda": 42.44, "Direita": 42.88},
    "BB": {"Esquerda": 17.02, "Direita": 24.79},
    "CC": {"Esquerda": 13.41, "Direita": 14.11},
    "AG": {"Esquerda": 25.86, "Direita": 59.89},
    "BG": {"Esquerda": 7.27,  "Direita": 36.47},
    "CG": {"Esquerda": 10.08, "Direita": 17.05},
    "FF": {"Esquerda": 24.52, "Direita": 30.86},
    "NN": {"Esquerda": 13.11, "Direita": 25.53},
    "DD": {"Esquerda": 11.64, "Direita": 11.58},
    "FJ": {"Esquerda": 23.49, "Direita": 28.36},
    "NF": {"Esquerda": 7.77,  "Direita": 15.88},
    "DF": {"Esquerda": 12.98, "Direita": 13.91},
    "RR": {"Esquerda": 13.08, "Direita": 47.28},
    "PP": {"Esquerda": 6.41,  "Direita": 5.31},
    "II": {"Esquerda": 5.17,  "Direita": 19.42},
    "RE": {"Esquerda": 31.10, "Direita": 32.47},
    "PE": {"Esquerda": 19.55, "Direita": 27.26},
    "IE": {"Esquerda": 43.41, "Direita": 24.02}
}

def calcular_similaridade_tempos(df_resultados):
    colunas_objetos = [c for c in df_resultados.columns if c.lower() not in ['vídeo / par', 'floor', 'chao']]
    linhas_comparacao = []
    def calc_sim_estrita(script, manual):
        if script == 0 and manual == 0: return 100.0
        if script == 0 or manual == 0: return 0.0
        return round((min(script, manual) / max(script, manual)) * 100, 2)

    for index, row in df_resultados.iterrows():
        nome_arquivo = str(row['Vídeo / Par'])
        nome_limpo = nome_arquivo.replace('_NOVO', '').replace('_TCR', '')
        match_composto = re.search(r'([A-Za-z\-]+)_(\d+)', nome_limpo)
        par = None
        if match_composto:
            sequencia_pares = match_composto.group(1).split('-')
            indice_real = int(match_composto.group(2)) - 1
            if 0 <= indice_real < len(sequencia_pares):
                par = sequencia_pares[indice_real]
        else:
            match_simples = re.search(r'^([A-Z]{2})', nome_limpo)
            if match_simples:
                par = match_simples.group(1)

        if par and par in tempos_manuais:
            tempos_script_validos = []
            for col in colunas_objetos:
                if row[col] > 0:
                    tempos_script_validos.append(row[col])
            tempos_script_validos = sorted(tempos_script_validos, reverse=True)[:2]
            while len(tempos_script_validos) < 2:
                tempos_script_validos.append(0.0)
            tempo_manual_esq = tempos_manuais[par]["Esquerda"]
            tempo_manual_dir = tempos_manuais[par]["Direita"]
            sim_A1 = calc_sim_estrita(tempos_script_validos[0], tempo_manual_esq)
            sim_A2 = calc_sim_estrita(tempos_script_validos[1], tempo_manual_dir)
            media_A = (sim_A1 + sim_A2) / 2
            sim_B1 = calc_sim_estrita(tempos_script_validos[0], tempo_manual_dir)
            sim_B2 = calc_sim_estrita(tempos_script_validos[1], tempo_manual_esq)
            media_B = (sim_B1 + sim_B2) / 2

            if media_A >= media_B:
                t_script_esq, t_script_dir = tempos_script_validos[0], tempos_script_validos[1]
                sim_esq, sim_dir = sim_A1, sim_A2
            else:
                t_script_esq, t_script_dir = tempos_script_validos[1], tempos_script_validos[0]
                sim_esq, sim_dir = sim_B2, sim_B1
            linhas_comparacao.append({
                "Par Real": par,
                "Arquivo": nome_arquivo,
                "Script Esq (s)": t_script_esq,
                "Manual Esq (s)": tempo_manual_esq,
                "Semelhança Esq (%)": sim_esq,
                "|": "|",
                "Script Dir (s)": t_script_dir,
                "Manual Dir (s)": tempo_manual_dir,
                "Semelhança Dir (%)": sim_dir
            })
    df_comparacao = pd.DataFrame(linhas_comparacao)
    if not df_comparacao.empty:
        display(df_comparacao)
        DIRETORIO_RESULTADOS = '/content/drive/MyDrive/DLC_VS_TOPSCAN_2026/RESULTADOS/'
        os.makedirs(DIRETORIO_RESULTADOS, exist_ok=True)
        caminho_salvar = os.path.join(DIRETORIO_RESULTADOS, 'Validacao_Semelhanca_TopScan.xlsx')
        df_comparacao.to_excel(caminho_salvar, index=False)

if 'df_resultados' in locals() or 'df_resultados' in globals():
    calcular_similaridade_tempos(df_resultados)
